# analysis.cooccurrence-habitat

In this notebook, we aim to understand:
- how cooccurrences are distributed across sites
- how cooccurrences could be related with site species richness

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from daforfer import DaforferDB
import matplotlib.pyplot as plt
import networkx as nx
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
si = DaforferDB(conf['si'])
si.toc()

## Load data

In [ ]:
alpha_diversity = db.conn.sql('SELECT * FROM D_Site_level_div').df()

In [ ]:
g = sns.catplot(alpha_diversity, x='site', y='total_cooccurrences', height=2.0, aspect=2.0, kind='bar', hue='habitat', palette=conf['habitat_palette'], edgecolor='black', linewidth=0.65)
# g.axes[0, 0].set_yscale('log')
g.set_ylabels("Number of cooccurrences")
g.set_xlabels("Site code")
g.set_xticklabels(rotation=90)
g.savefig("figures/barplot.cooccurrencesbysite.colbyhabitat.svg")

In [ ]:
g = sns.catplot(alpha_diversity, x='disturbed', y='total_cooccurrences', height=2.0, aspect=1.0, hue='disturbed', linewidth=0.65, kind='box', whis=1000)
# g.axes[0, 0].set_yscale('log')
g.set_ylabels("Number of cooccurrences")
g.set_xlabels("Habitat")
# g.set_xticklabels(rotation=90)
g.savefig("figures/boxplot.cooccurrences.bydisturbance.svg")

In [ ]:
g = sns.catplot(alpha_diversity, x='habitat', y='total_cooccurrences', height=2.0, aspect=1.25, hue='habitat', palette=conf['habitat_palette'], linewidth=0.65, kind='box', whis=1000)
# g.axes[0, 0].set_yscale('log')
g.set_ylabels("Number of cooccurrences")
g.set_xlabels("Habitat")
# g.set_xticklabels(rotation=90)
g.savefig("figures/boxplot.cooccurrences.byhabitat.svg")

In [ ]:
stats.kruskal(
    alpha_diversity.query('habitat == "Crop"')['total_cooccurrences'],
    alpha_diversity.query('habitat == "Edge"')['total_cooccurrences'],
    alpha_diversity.query('habitat == "Oak"')['total_cooccurrences'],
    alpha_diversity.query('habitat == "Wasteland"')['total_cooccurrences'],
)

In [ ]:
stats.mannwhitneyu(
    alpha_diversity.query('habitat == "Crop" or habitat == "Edge"')['total_cooccurrences'],
    alpha_diversity.query('habitat == "Oak" or habitat == "Wasteland"')['total_cooccurrences']
)

### Post-hoc analysis

In [ ]:
post_hoc_stats = []

for h1 in ['Crop', 'Edge', 'Wasteland', 'Oak']:
    for h2 in ['Crop', 'Edge', 'Wasteland', 'Oak']:
        if h1 != h2:
            kw_h, pval = stats.mannwhitneyu(
                alpha_diversity.query('habitat == "{0}"'.format(h1))['total_cooccurrences'].values,
                alpha_diversity.query('habitat == "{0}"'.format(h2))['total_cooccurrences'].values,
                alternative='less'
            )
            significative = pval < 0.05
            post_hoc_stats.append(
                {'group_1': h1, 'group_2': h2, 'U': kw_h, 'p-val': pval, 'sign': significative}
            )
post_hoc_stats = pd.DataFrame.from_records(post_hoc_stats)

db.save_dataframe(
    df=post_hoc_stats, table_name="T_coocByHabitat",
    description="Post-Hoc Mann Whitney U analysis on number of cooccurrennt pairs by habitat, group1 < group2"
)

si.save_dataframe(
    df=post_hoc_stats, table_name="TableS7",
    description="Mann-Whitney U post-hoc test on site-diversity by habitat"
)

post_hoc_stats

In [ ]:
post_hoc_stats.pivot(index='group_1', columns='group_2', values='p-val')

In [ ]:
post_hoc_stats.pivot(index='group_1', columns='group_2', values='U').round(4)

## Correlation between site-alpha-diversity and site-number of cooccurrences

In [ ]:
# alpha_diversity = pd.read_csv("output/diversity.all.csv", sep=";")
# alpha_diversity = db.conn.sql('SELECT * FROM D_ADAllOrganismsSite').df().drop(columns=['disturbed'])
# alpha_diversity = pd.merge(alpha_diversity, site_cooccurrences, on=['site', 'habitat'])
# db.save_dataframe(
#     alpha_diversity, table_name="Alpha_diversity_bysite_cooccurrences",
#     description="Alpha diveristy and cooccurrences by site"
# )
# alpha_diversity

In [ ]:
g = sns.lmplot(alpha_diversity, x='species_richness_bact', y='total_cooccurrences', height=2.0)
g.set_xlabels("SR (bact)")
g.set_ylabels("Coccurrence per site")

In [ ]:
g = sns.lmplot(alpha_diversity, x='species_richness_vir', y='total_cooccurrences', height=2.0)
g.set_xlabels("SR (vir)")
g.set_ylabels("Coccurrence per site")

In [ ]:
g = sns.lmplot(alpha_diversity, x='species_richness_plant', y='total_cooccurrences', height=2.0)
g.set_xlabels("SR (plant)")
g.set_ylabels("Coccurrence per site")

In [ ]:
def reg2dict(x, level):

    return {"level": level, "p-value": x.pvalue, "r-value": x.rvalue, "slope": x.slope}
regression_tests = pd.DataFrame.from_records([
    reg2dict(stats.linregress(alpha_diversity['species_richness_bact'], alpha_diversity['total_cooccurrences']), "Bact"),
    reg2dict(stats.linregress(alpha_diversity['species_richness_vir'], alpha_diversity['total_cooccurrences']), "Virus"),
    reg2dict(stats.linregress(alpha_diversity['species_richness_plant'], alpha_diversity['total_cooccurrences']), "Plant")
])

db.save_dataframe(
    regression_tests, table_name='T_coocDivBySite',
    description='Correlation tests between species richness and number of coocurrences detected at each site'
)
regression_tests

In [17]:
db.conn.close()
si.conn.close()